# Sql Avanzado

Se utilizara el mismo dataset de "SQL Intermedio" chinook.

In [2]:
%load_ext sql

%sql sqlite:///data/sql/Chinook_Sqlite.sqlite

Connecting to 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

In [3]:
import sqlite3
import pandas as pd
import urllib.request

url = 'https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite'
urllib.request.urlretrieve(url, 'data/sql/Chinook_Sqlite.sqlite')

conn = sqlite3.connect('data/sql/Chinook_Sqlite.sqlite')

tables_query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql(tables_query, conn)
tables



,name
0,Album
1,Artist
2,Customer
3,Employee
4,Genre
5,Invoice
6,InvoiceLine
7,MediaType
8,Playlist
9,PlaylistTrack


In [4]:
%%sql
PRAGMA table_info("Employee") 

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

cid,name,type,notnull,dflt_value,pk
0,EmployeeId,INTEGER,1,None,1
1,LastName,NVARCHAR(20),1,None,0
2,FirstName,NVARCHAR(20),1,None,0
3,Title,NVARCHAR(30),0,None,0
4,ReportsTo,INTEGER,0,None,0
5,BirthDate,DATETIME,0,None,0
6,HireDate,DATETIME,0,None,0
7,Address,NVARCHAR(70),0,None,0
8,City,NVARCHAR(40),0,None,0
9,State,NVARCHAR(40),0,None,0


## Cambiar el tipo de dato de una columna
Sirve para aquellos datos que no fueron guardados de la mejor manera posible, como numeros o fechas. Esto puede llevar a problemas sumando datos o realizando funciones que da error por el tipo de dato incorrecto.

Para realizar esta conversion se utiliza "CAST" o "CONVERT", estos se pueden utilizar de la siguiente manera: 
- CAST(columna AS integer)
- columna::integer - Esto funciona en PostgreSQL

Estas dos formas obtienen el mismo resultado, convierten el tipo de dato de la columna a integer. Claro puede ser cualquier otro tipo de dato.

In [5]:
%%sql
SELECT
    EmployeeId Employee_integer,
    typeof(EmployeeId) AS tipo_original,
    CAST(EmployeeId AS VARCHAR) AS Employee_varchar,
    typeof(CAST(EmployeeId AS VARCHAR)) AS tipo_convertido
FROM Employee
LIMIT 1;
    

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Employee_integer,tipo_original,Employee_varchar,tipo_convertido
1,integer,1,text


## Formato de fechas

Tener el formato correcto de fechas puede ayudar a realizar ciertas operaciones:
- fecha - fecha = intervalo
- fecha + intervalo = otra fecha
- fecha - intervalo = otra fecha

In [6]:
%%sql
SELECT 
    FirstName,
    LastName,
    BirthDate,
    HireDate,
    JULIANDAY(HireDate) - JULIANDAY(BirthDate) AS dias_hasta_contratacion
FROM Employee;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

FirstName,LastName,BirthDate,HireDate,dias_hasta_contratacion
Andrew,Adams,1962-02-18 00:00:00,2002-08-14 00:00:00,14787.0
Nancy,Edwards,1958-12-08 00:00:00,2002-05-01 00:00:00,15850.0
Jane,Peacock,1973-08-29 00:00:00,2002-04-01 00:00:00,10442.0
Margaret,Park,1947-09-19 00:00:00,2003-05-03 00:00:00,20315.0
Steve,Johnson,1965-03-03 00:00:00,2003-10-17 00:00:00,14107.0
Michael,Mitchell,1973-07-01 00:00:00,2003-10-17 00:00:00,11065.0
Robert,King,1970-05-29 00:00:00,2004-01-02 00:00:00,12271.0
Laura,Callahan,1968-01-09 00:00:00,2004-03-04 00:00:00,13204.0


JULIANDAY() convierte una fecha a un numero y al restar las dos fechas transformadas a numero se obtiene la diferencia en dias.

In [7]:
%%sql
SELECT 
    FirstName,
    HireDate,
    DATE(HireDate, '+7 days') AS una_semana_despues
FROM Employee;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

FirstName,HireDate,una_semana_despues
Andrew,2002-08-14 00:00:00,2002-08-21
Nancy,2002-05-01 00:00:00,2002-05-08
Jane,2002-04-01 00:00:00,2002-04-08
Margaret,2003-05-03 00:00:00,2003-05-10
Steve,2003-10-17 00:00:00,2003-10-24
Michael,2003-10-17 00:00:00,2003-10-24
Robert,2004-01-02 00:00:00,2004-01-09
Laura,2004-03-04 00:00:00,2004-03-11


SQLite usa DATE(fecha, modificacion) como por ejemplo de modificacion:
- +7 days
- +1 month
- -3 years

In [8]:
%%sql
SELECT 
    FirstName,
    HireDate,
    JULIANDAY('now') - JULIANDAY(HireDate) AS dias_desde_contratacion
FROM Employee;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

FirstName,HireDate,dias_desde_contratacion
Andrew,2002-08-14 00:00:00,8767.885547025595
Nancy,2002-05-01 00:00:00,8872.885547025595
Jane,2002-04-01 00:00:00,8902.885547025595
Margaret,2003-05-03 00:00:00,8505.885547025595
Steve,2003-10-17 00:00:00,8338.885547025595
Michael,2003-10-17 00:00:00,8338.885547025595
Robert,2004-01-02 00:00:00,8261.885547025595
Laura,2004-03-04 00:00:00,8199.885547025595


'now' es una palabra clave especial de SQLite que representa la fecha/hora actual del sistema.

## Limpiar datos

### Length() y Substr()

Con "Length()" Se obtiene el largo total de una palabra y con "Substr()" se puede obtener cierta cantidad de caracteres segun se necesite

In [9]:
%%sql
SELECT 
    FirstName,
    SUBSTR(FirstName, 1, 3) AS primeros_3,          
    SUBSTR(FirstName, -3, 3) AS ultimos_3,          
    LENGTH(FirstName) AS largo
FROM Customer;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

FirstName,primeros_3,ultimos_3,largo
Luís,Luí,uís,4
Leonie,Leo,nie,6
François,Fra,ois,8
Bjørn,Bjø,ørn,5
František,Fra,šek,9
Helena,Hel,ena,6
Astrid,Ast,rid,6
Daan,Daa,aan,4
Kara,Kar,ara,4
Eduardo,Edu,rdo,7


### Trim() y sus variantes

- **Trim()**: Sirve para quitar los espacios de ambos lados.
- **LTrim()**: Sirve para quitar los espacios del lado izquierdo.
- **RTrim()**: Sirve para quitar los espacios del lado derecho.
- **Trim(palabra, caracter)**: Quita el caracter especifico a ambos lados de la palabra.

In [10]:
%%sql
SELECT 
    TRIM('  Hola Mundo  ') AS limpio,         
    LTRIM('  Hola')  AS solo_izquierda,       
    RTRIM('Hola  ')  AS solo_derecha,        
    TRIM('***Hola***', '*') AS quita_asteriscos; 

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

limpio,solo_izquierda,solo_derecha,quita_asteriscos
Hola Mundo,Hola,Hola,Hola


### Instr()
Devuelve la posicion de un subtring dentro de un texto.
- Instr(texto, substring)

In [11]:
%%sql
SELECT 
    Email,
    INSTR(Email, '@') AS posicion_arroba
FROM Customer
LIMIT 5;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Email,posicion_arroba
luisg@embraer.com.br,6
leonekohler@surfeu.de,12
ftremblay@gmail.com,10
bjorn.hansen@yahoo.no,13
frantisekw@jetbrains.com,11


## Operador ||
Sirve para concatenar palabras.

In [12]:
%%sql
SELECT 
    FirstName || ' ' || LastName AS nombre_completo
FROM Customer;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

nombre_completo
Luís Gonçalves
Leonie Köhler
François Tremblay
Bjørn Hansen
František Wichterlová
Helena Holý
Astrid Gruber
Daan Peeters
Kara Nielsen
Eduardo Martins


### Upper() y Lower()

In [13]:
%%sql
SELECT 
    Name,
    UPPER(Name) AS mayusculas,
    LOWER(Name) AS minusculas
FROM Artist;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Name,mayusculas,minusculas
AC/DC,AC/DC,ac/dc
Accept,ACCEPT,accept
Aerosmith,AEROSMITH,aerosmith
Alanis Morissette,ALANIS MORISSETTE,alanis morissette
Alice In Chains,ALICE IN CHAINS,alice in chains
Antônio Carlos Jobim,ANTôNIO CARLOS JOBIM,antônio carlos jobim
Apocalyptica,APOCALYPTICA,apocalyptica
Audioslave,AUDIOSLAVE,audioslave
BackBeat,BACKBEAT,backbeat
Billy Cobham,BILLY COBHAM,billy cobham


### Strftime()
Sirve para descomponer fechas(año, mes, dia) o redondear a una unidad de tiempo.

In [14]:
%%sql
SELECT 
    HireDate,
    STRFTIME('%Y', HireDate) AS año,
    STRFTIME('%m', HireDate) AS mes,
    STRFTIME('%d', HireDate) AS dia,
    STRFTIME('%w', HireDate) AS dia_semana  -- 0=domingo, 6=sábado
FROM Employee;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

HireDate,año,mes,dia,dia_semana
2002-08-14 00:00:00,2002,08,14,3
2002-05-01 00:00:00,2002,05,01,3
2002-04-01 00:00:00,2002,04,01,1
2003-05-03 00:00:00,2003,05,03,6
2003-10-17 00:00:00,2003,10,17,5
2003-10-17 00:00:00,2003,10,17,5
2004-01-02 00:00:00,2004,01,02,5
2004-03-04 00:00:00,2004,03,04,4


In [15]:
%%sql
SELECT 
    HireDate,
    DATE(HireDate, 'start of month') AS inicio_del_mes
FROM Employee;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

HireDate,inicio_del_mes
2002-08-14 00:00:00,2002-08-01
2002-05-01 00:00:00,2002-05-01
2002-04-01 00:00:00,2002-04-01
2003-05-03 00:00:00,2003-05-01
2003-10-17 00:00:00,2003-10-01
2003-10-17 00:00:00,2003-10-01
2004-01-02 00:00:00,2004-01-01
2004-03-04 00:00:00,2004-03-01


### Fecha y hora actual

In [16]:
%%sql
SELECT 
    CURRENT_DATE AS fecha_actual,
    CURRENT_TIME AS hora_actual,
    CURRENT_TIMESTAMP AS fecha_y_hora_actual;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

fecha_actual,hora_actual,fecha_y_hora_actual
2026-08-15,21:15:11,2026-08-15 21:15:11


### Coalesce
Reemplaza valores nulos por un valor por defecto.

In [17]:
%%sql
SELECT 
    ar.Name AS Artista,
    COALESCE(a.Title, 'Sin álbum') AS Album
FROM Artist AS ar
LEFT JOIN Album AS a ON a.ArtistId = ar.ArtistId;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Artista,Album
AC/DC,For Those About To Rock We Salute You
AC/DC,Let There Be Rock
Accept,Balls to the Wall
Accept,Restless and Wild
Aerosmith,Big Ones
Alanis Morissette,Jagged Little Pill
Alice In Chains,Facelift
Antônio Carlos Jobim,Warner 25 Anos
Antônio Carlos Jobim,Chill: Brazil (Disc 2)
Apocalyptica,Plays Metallica By Four Cellos


### Ejercicio: Extraer dominio de mails.

In [18]:
%%sql
SELECT 
    Email,
    SUBSTR(Email, INSTR(Email, '@') + 1) AS dominio
FROM Customer;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Email,dominio
luisg@embraer.com.br,embraer.com.br
leonekohler@surfeu.de,surfeu.de
ftremblay@gmail.com,gmail.com
bjorn.hansen@yahoo.no,yahoo.no
frantisekw@jetbrains.com,jetbrains.com
hholy@gmail.com,gmail.com
astrid.gruber@apple.at,apple.at
daan_peeters@apple.be,apple.be
kara.nielsen@jubii.dk,jubii.dk
eduardo@woodstock.com.br,woodstock.com.br


## Subconsultas
Tambien llamada consulta anidada, es una herramienta para realizar operaciones en varios pasos.

### Subconsulta en From

Primero se realiza la consulta que esta dentro del "FROM" y lego el resto del codigo se trabaja en base a lo que la primera consulta trajo.

In [19]:
%%sql

SELECT sub.*
FROM (
        SELECT *
        FROM Employee
        Where City = 'Calgary'
     ) AS sub
WHERE Title = 'Sales Support Agent';

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

EmployeeId,LastName,FirstName,Title,ReportsTo,BirthDate,HireDate,Address,City,State,Country,PostalCode,Phone,Fax,Email
3,Peacock,Jane,Sales Support Agent,2,1973-08-29 00:00:00,2002-04-01 00:00:00,1111 6 Ave SW,Calgary,AB,Canada,T2P 5M5,+1 (403) 262-3443,+1 (403) 262-6712,jane@chinookcorp.com
4,Park,Margaret,Sales Support Agent,2,1947-09-19 00:00:00,2003-05-03 00:00:00,683 10 Street SW,Calgary,AB,Canada,T2P 5G3,+1 (403) 263-4423,+1 (403) 263-4289,margaret@chinookcorp.com
5,Johnson,Steve,Sales Support Agent,2,1965-03-03 00:00:00,2003-10-17 00:00:00,7727B 41 Ave,Calgary,AB,Canada,T3B 1Y7,1 (780) 836-9987,1 (780) 836-9543,steve@chinookcorp.com


### Agregacion en multiples etapas

¿Cual es el gasto promedio por factura, para cada pais, considerenado solo los paises con mas de 5 facturas?

Etapas para responder:
1. Calcular el todal de cada factura individual (sumando sus lineas de detalle).
2. Promediar esos totales por pais.

##### 1. Subconsulta interna: Total por factura.

In [20]:
%%sql
SELECT 
    i.InvoiceId,
    i.BillingCountry,
    SUM(il.UnitPrice * il.Quantity) AS total_factura
FROM Invoice AS i
JOIN InvoiceLine AS il ON il.InvoiceId = i.InvoiceId
GROUP BY i.InvoiceId, i.BillingCountry

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

InvoiceId,BillingCountry,total_factura
1,Germany,1.98
2,Norway,3.96
3,Belgium,5.9399999999999995
4,Canada,8.91
5,USA,13.86
6,Germany,0.99
7,Germany,1.98
8,France,1.98
9,France,3.96
10,Ireland,5.9399999999999995


##### 2. Consulta externa: Promedio por pais, usando el resultado anterior.

In [21]:
%%sql

SELECT 
    sub.BillingCountry AS Pais,
    COUNT(sub.InvoiceId) AS Cantidad_Facturas,
    AVG(sub.total_factura) AS Promedio_Por_Factura
FROM (
    SELECT 
        i.InvoiceId,
        i.BillingCountry,
        SUM(il.UnitPrice * il.Quantity) AS total_factura
    FROM Invoice AS i
    JOIN InvoiceLine AS il ON il.InvoiceId = i.InvoiceId
    GROUP BY i.InvoiceId, i.BillingCountry
) AS sub
GROUP BY sub.BillingCountry
HAVING COUNT(sub.InvoiceId) > 5
ORDER BY Promedio_Por_Factura DESC;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Pais,Cantidad_Facturas,Promedio_Por_Factura
Chile,7,6.659999999999999
Ireland,7,6.517142857142857
Hungary,7,6.517142857142857
Czech Republic,14,6.445714285714286
Austria,7,6.088571428571428
Finland,7,5.945714285714286
Netherlands,7,5.8028571428571425
India,13,5.789230769230769
USA,91,5.747912087912088
Norway,7,5.659999999999999


##### Comparacion con CTE (Common Table Expression)

In [24]:
%%sql

WITH totales_por_factura AS (
    SELECT 
        i.InvoiceId,
        i.BillingCountry,
        SUM(il.UnitPrice * il.Quantity) AS total_factura
    FROM Invoice AS i
    JOIN InvoiceLine AS il ON il.InvoiceId = i.InvoiceId
    GROUP BY i.InvoiceId, i.BillingCountry
)

SELECT 
    BillingCountry AS Pais,
    COUNT(InvoiceId) AS Cantidad_Por_Factura,
    AVG(Total_factura) AS Promedio_Por_Factura
FROM totales_por_factura
GROUP BY BillingCountry
HAVING COUNT(InvoiceId) > 5
ORDER BY Promedio_Por_Factura DESC;

Running query in 'sqlite:///data/sql/Chinook_Sqlite.sqlite'

Pais,Cantidad_Por_Factura,Promedio_Por_Factura
Chile,7,6.659999999999999
Ireland,7,6.517142857142857
Hungary,7,6.517142857142857
Czech Republic,14,6.445714285714286
Austria,7,6.088571428571428
Finland,7,5.945714285714286
Netherlands,7,5.8028571428571425
India,13,5.789230769230769
USA,91,5.747912087912088
Norway,7,5.659999999999999


### Subconsultas en logica condicional (Where, Join, On, Case)


### uniendo subconsultas


### Subconsulta con union